# Multiclass Classification

In [ ]:
import os
import warnings
import logging

# configure logging
logging.basicConfig(
    level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

# get warning filter policy from the environment variables
# set to "ignore" for rendering the HTMLs, or to "once" otherwise
WARNING_FILTER_POLICY = os.getenv("WARNING_FILTER_POLICY", "once")
logger.info(f"{WARNING_FILTER_POLICY = }")
warnings.filterwarnings(WARNING_FILTER_POLICY)

In [ ]:
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
import shap

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, GridSearchCV
from statsmodels.stats.outliers_influence import variance_inflation_factor
from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
from utils.constants import RANDOM_SEED
from utils.common import (
    get_data_folder_path,
    set_plotting_config,
    plot_boxplot_by_class,
    plot_correlation_matrix,
)
from utils.evals import (
    describe_input_features,
    plot_confusion_matrix,
    plot_target_rate,
    compute_multiclass_classification_metrics,
    build_coefficients_table,
    plot_coefficients_values,
    plot_coefficients_significance,
    plot_eval_metrics_xgb,
    plot_gain_metric_xgb,
    plot_shap_importance,
    plot_shap_beeswarm,
    plot_roc_curve,
)
from utils.feature_selection import run_feature_selection_steps

In [ ]:
# plots configuration
sns.set_style("darkgrid")
sns.set_palette("colorblind")
set_plotting_config()
%matplotlib inline

## 1. Load Data

In this notebook, we will use the **Fetal Health Dataset**. This dataset comprises 2126 records of features from Cardiotocogram exams, classified by experts into Normal, Suspect, and Pathological to assess fetal health and help reduce child and maternal mortality.

Sources:
1. Kaggle: https://www.kaggle.com/datasets/andrewmvd/fetal-health-classification
2. Original article: https://www.ncbi.nlm.nih.gov/pmc/articles/PMC68223152

In [ ]:
data_path = get_data_folder_path()

df_input = pd.read_csv(os.path.join(data_path, "fetal_health.csv"))
df_input.columns = [col.replace(" ", "_") for col in df_input.columns]

## 2. Process Data

### Target column

Fetal health (target column) can have the following values:
- 1: Normal
- 2: Suspect
- 3: Pathological

However, XGBoost expects 0-indexed positive integers for the classes. Therefore, we will use the following values in this notebook:
- 0: Normal
- 1: Suspect
- 2: Pathological

In [ ]:
target_col = "fetal_health"
target_classes_dict = {
    0: "Normal",
    1: "Suspect",
    2: "Pathological",
}
test_size = 0.20

In [ ]:
# convert target column to integer
df_input[target_col] = df_input[target_col].astype(np.int8) - np.int8(1)  # subtract 1 to make it 0-indexed

### Train test split

In [ ]:
df_input_train, df_input_test = train_test_split(
    df_input,
    test_size=test_size,
    stratify=df_input[target_col],
    random_state=RANDOM_SEED,
)

In [ ]:
pd.concat([
    pd.Series(target_classes_dict, name="label"),
    df_input_train[target_col].value_counts(dropna=False, normalize=False).rename("train_target_count"),
    df_input_train[target_col].value_counts(dropna=False, normalize=True).rename("train_target_pct"),
    df_input_test[target_col].value_counts(dropna=False, normalize=False).rename("test_target_count"),
    df_input_test[target_col].value_counts(dropna=False, normalize=True).rename("test_target_pct"),
], axis=1)

In [ ]:
describe_input_features(df_input, df_input_train, df_input_test)

### Scaling (Standardization)

In [ ]:
# Standardize training and test data
stdscaler = StandardScaler()

# training data
X_train_all = (
    pd.DataFrame(
        # fit scaler on training data (and then transform training data)
        data=stdscaler.fit_transform(df_input_train),
        columns=df_input_train.columns,
        index=df_input_train.index
    )
    # remove target from the model input features table
    .drop(columns=[target_col])
)
y_train = df_input_train[target_col]
y_train_ohe = pd.get_dummies(y_train, dtype=np.int8)  # one-hot encoding for plots

# test data
y_test = df_input_test[target_col]
X_test_all = (
    pd.DataFrame(
        # use scaler fitted on training data to transform test data
        data=stdscaler.transform(df_input_test),
        columns=df_input_test.columns,
        index=df_input_test.index
    )
    # remove target from the model input features table
    .drop(columns=[target_col])
)
y_test_ohe = pd.get_dummies(y_test, dtype=np.int8)  # one-hot encoding for plots

## 3. Exploratory Data Analysis (EDA)

### Boxplots by Target Class

In [ ]:
display(
    plot_boxplot_by_class(
        df_input=df_input_train,  # use only training data to avoid bias in test results
        class_col=target_col,
        class_mapping=target_classes_dict,
        plots_per_line=5,
        title="Features in input dataset",
    )
)

### Pearson's Correlation

In [ ]:
display(
    plot_correlation_matrix(
        # use only training data to avoid bias in test results
        df=df_input_train, method="pearson", fig_height=10
    )
)

## 4. Feature Selection

In [ ]:
fs_steps = {
    "manual": {
        "cols_to_exclude": [
            "severe_decelerations",
        ]
    },
    "null_variance": None,
    "correlation": {"threshold": 0.75},
    "vif": {"threshold": 2},
    "l1_regularization": {
        "problem_type": "classification",
        "train_test_split_params": {"test_size": test_size},
        "logspace_search": {"start": -5, "stop": 1, "num": 20, "base": 10},
        # tolerance over minimum error with which to search for the best model
        "error_tolerance_pct": 0.05,
        # minimum features to keep in final selection
        "min_feats_to_keep": 3,
        "random_seed": RANDOM_SEED,
    },
}

In [ ]:
# run Feature Selection separately for each class as binary classifications
selected_feats_ovr = {}
fs_tables_ovr = {}

for clss, label in target_classes_dict.items():
    logger.info(f"Running Feature Selection for Class '{label}' (vs Rest)")
    selected_feats_ovr[clss], fs_tables_ovr[clss] = run_feature_selection_steps(
        # use only training data to avoid bias in test results
        X=X_train_all,
        y=y_train_ohe[clss],
        fs_steps=fs_steps
    )
    logger.info("-" * 100)

In [ ]:
# keep only the features that were selected for at least 2 classes
MIN_NUM_SELECTIONS = 2

classes_intersections = []
for classes_group in itertools.combinations(selected_feats_ovr.keys(), MIN_NUM_SELECTIONS):
    classes_intersections.append(
        set.intersection(*[set(selected_feats_ovr[clss]) for clss in classes_group])
    )
    
selected_feats = list(set.union(*classes_intersections))
print(f"Final selection ({len(selected_feats)} features selected):")
for feat in sorted(selected_feats):
    print(f"  - {feat}")

In [ ]:
# build model input datasets
X_train = X_train_all[selected_feats]
X_test = X_test_all[selected_feats]

### Correlation check


In [ ]:
display(
    plot_correlation_matrix(
        # use only training data to avoid bias in test results
        df=df_input_train[selected_feats + [target_col]], method="pearson", fig_height=5
    )
)

### Multicollinearity check


In [ ]:
# compute the Variance Inflation Factor (VIF) for each feature
df_vif = pd.DataFrame(
    data=[variance_inflation_factor(X_train.values, i) for i in range(len(selected_feats))],
    index=selected_feats,
    columns=["VIF"]
).sort_values("VIF", ascending=False)

df_vif

## 5. Classifier Model

### Select classifier: Logistic Regression or XGBoost

In [ ]:
# MODEL_SELECTION = "logistic_regression"
MODEL_SELECTION = "xgboost"

model_selection_error = ValueError(
    "'MODEL_SELECTION' must be either 'logistic_regression' or 'xgboost'. "
    f"Got {MODEL_SELECTION} instead."
)

### Hyperparameter tuning with K-Fold Cross Validation

For a detailed explanation of XGBoost's parameters, refer to: https://www.kaggle.com/code/prashant111/a-guide-on-xgboost-hyperparameters-tuning/notebook

In [ ]:
if MODEL_SELECTION == "logistic_regression":
    Estimator = LogisticRegression
    cv_search_space = {
        "penalty": ["l1", "l2", "elasticnet"],
        "solver": ["saga"],
        "C": np.logspace(-3, 1, num=9, base=10.0),
        "class_weight": [None],
    }
elif MODEL_SELECTION == "xgboost":
    Estimator = XGBClassifier
    cv_search_space = {
        "objective": ["multi:softmax"],
        'num_class': [len(target_classes_dict)],
        "n_estimators": [30, 40, 50],
        "learning_rate": [0.1],
        "max_depth": [3, 4, 6],
        "min_child_weight": [2, 4],
        "gamma": [0, 0.5],
        "alpha": [0, 0.3],
        "scale_pos_weight": [1],
        "lambda": [1],
        ## "subsample": [0.8, 1.0],
        ## "colsample_bytree": [0.8, 1.0],
        "verbosity": [0],
    }
else:
    raise model_selection_error

For the full list of scikit-learn's scoring string names, refer to: https://scikit-learn.org/stable/modules/model_evaluation.html#string-name-scorers

In [ ]:
cv_scoring_metrics = {
    "accuracy": "Accuracy",
    "precision_macro": "Precision (macro)",
    "recall_macro": "Recall (macro)",
    "f1_macro": "F1 Score (macro)",
    "precision_weighted": "Precision (weighted)",
    "recall_weighted": "Recall (weighted)",
    "f1_weighted": "F1 Score (weighted)",
    "roc_auc_ovr": "ROC AUC One-vs-Rest (macro)",
    "roc_auc_ovo": "ROC AUC One-vs-One (macro)",
    "roc_auc_ovr_weighted": "ROC AUC One-vs-Rest (weighted)",
    "roc_auc_ovo_weighted": "ROC AUC One-vs-One (weighted)",
}
refit_metric = "f1_weighted"  # metric to optimize for the final model

In [ ]:
%%time
# define evaluation
kfold_cv = RepeatedStratifiedKFold(n_splits=3, n_repeats=1, random_state=RANDOM_SEED)
# define search
grid_search = GridSearchCV(
    estimator=Estimator(),
    param_grid=cv_search_space,
    scoring=list(cv_scoring_metrics.keys()),
    cv=kfold_cv,
    refit=refit_metric,
    verbose=1,
)
# execute search
result_cv = grid_search.fit(X_train, y_train)

In [ ]:
print("Grid Search CV Best Model - Scoring Metrics (averaging method):")
for i, (metric_key, metric_name) in enumerate(cv_scoring_metrics.items(), start=1):
    print(
        f" {str(i) + ".":>3} {metric_name:.<31} "
        f"{result_cv.cv_results_[f"mean_test_{metric_key}"][result_cv.best_index_]:.3f}"
    )
print(f"\nBest Hyperparameters: {result_cv.best_params_}")

### Final Model

In [ ]:
# instantiate model with best hyperparameters and additional kwargs
if MODEL_SELECTION == "logistic_regression":
    model_kwargs = dict()
    model_fit_kwargs = dict()
elif MODEL_SELECTION == "xgboost":
    eval_metrics = dict(
        mlogloss="Binary Cross-entropy Loss (Log-loss)",
        merror="Binary Classification Error Rate",
        auc="ROC AUC",
    )
    model_kwargs = dict(eval_metric=list(eval_metrics.keys()))
    model_fit_kwargs = dict(
        eval_set=[(X_train, y_train), (X_test, y_test)],
        verbose=False
    )
else:
    raise model_selection_error
    
model = Estimator(**result_cv.best_params_, **model_kwargs, random_state=RANDOM_SEED)

In [ ]:
# Fit model and make predictions
model.fit(X_train, y_train, **model_fit_kwargs)
# Make probabilities predictions
y_pred_proba_train = pd.DataFrame(
    data=model.predict_proba(X_train), columns=model.classes_, index=X_train.index
)
y_pred_proba = pd.DataFrame(
    data=model.predict_proba(X_test), columns=model.classes_, index=X_test.index
)
# Make class predictions
y_pred_train = pd.Series(
    data=model.predict(X_train), index=X_train.index, name=target_col
)
y_pred = pd.Series(
    data=model.predict(X_test), index=X_test.index, name=target_col
)

In [ ]:
if MODEL_SELECTION == "xgboost":
    display(plot_eval_metrics_xgb(model.evals_result(), eval_metrics))

**Plot target rate per group of predicted probability**

A good model should have increasing target rate for each group of predicted probability (e.g. quartiles, deciles)

In [ ]:
for clss, label in target_classes_dict.items():
    title = f"Class '{label}': Target rate per group of predicted probability"
    display(
        plot_target_rate(y_test=y_test_ohe[clss], y_pred_proba=y_pred_proba[clss], title=title)
    )

### Feature Importance

- For Logistic Regression: coefficients values and statistical significance
- For XGBoost: SHAP analysis and Gain Metric

In [ ]:
if MODEL_SELECTION == "logistic_regression":
    for clss, coefficients, intercept in zip(
        model.classes_, model.coef_, model.intercept_
    ):
        label = f"Class '{target_classes_dict[clss]}'"
        print(label)
        df_coefficients = build_coefficients_table(
            coefficients=coefficients,
            intercept=intercept,
            X_train=X_train,
            y_pred_train=y_pred_proba_train[clss],
            y_train=y_train_ohe[clss],
            problem_type="classification",
        )
        display(
            plot_coefficients_values(
                df_coefficients,
                title=f"{label}: Coefficient Values with 95% CI (±1.96 Std Error)"
            ),
            plot_coefficients_significance(
                df_coefficients,
                log_scale=False,
                title=f"{label}: Coefficient Significance"
            ),
        )
    
elif MODEL_SELECTION == "xgboost":
    # compute SHAP values
    explainer = shap.Explainer(model)
    shap_values = explainer(X_test)
    # shap plots
    for i, clss in enumerate(model.classes_):
        label = f"Class '{target_classes_dict[clss]}'"
        print(label)
        display(
            plot_shap_importance(
                shap_values[:, :, i], title=f"{label}: SHAP Feature Importance"
            ),
            plot_shap_beeswarm(
                shap_values[:, :, i], title=f"{label}: SHAP Summary Plot"
            ),
            plot_gain_metric_xgb(
                model, X_test, title=f"{label}: XGBoost Feature Importance (Gain metric)"
            )
        )

else:
    raise model_selection_error

### Performance Metrics

In [ ]:
df_train_metrics = pd.Series(
    compute_multiclass_classification_metrics(y_train, y_pred_train, y_pred_proba_train)
).to_frame(name="Train Metrics")
df_test_metrics = pd.Series(
    compute_multiclass_classification_metrics(y_test, y_pred, y_pred_proba)
).to_frame(name="Test Metrics")

print("Final Model - Scoring Metrics on Train & Test Datasets:")
df_metrics = df_train_metrics.join(df_test_metrics)
display(df_metrics)

#### Confusion Matrix

In [ ]:
# Confusion Matrix
display(
    plot_confusion_matrix(
        y_test,
        y_pred,
        estimator=model,
        target_classes_dict=target_classes_dict,
        normalize="true",
    )
)

#### ROC AUC

In [ ]:
for clss, label in target_classes_dict.items():
    title = f"Class '{label}': ROC Curve One-vs-Rest"
    display(
        plot_roc_curve(
            y_true=y_test_ohe[clss], y_pred_proba=y_pred_proba[clss], title=title,
        )
    )